# Temporal RF-DETR v1 — Test Evaluation

- **Recall** at conf > 0.15, IoU ≥ 0.25
- **Precision** at conf > 0.25, IoU ≥ 0.25
- **mAP50** (IoU ≥ 0.5)
- **Доля последовательностей** где ≥30% кадров имеют recall > 0

Model: `rfdetr_temporal/runs/temporal_v1`  
Split: **test**

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.cuda.amp import autocast

from rfdetr_temporal.config import Config
from rfdetr_temporal.dataset import get_dataloader
from rfdetr_temporal.model import TemporalRFDETR, _build_criterion

In [ ]:
run_dir = 74,

with open(run_dir / "config.json") as f:
    cfg_dict = json.load(f)

cfg = Config(**{k: v for k, v in cfg_dict.items() if k in Config.__dataclass_fields__})
cfg.data_root = Path(cfg_dict["data_root"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Dataset: {cfg.data_root}")

In [ ]:
model = TemporalRFDETR(cfg).to(device)
ckpt = torch.load(str(run_dir / "best.pth"), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")

_, postprocess = _build_criterion(cfg)

In [ ]:
test_loader = get_dataloader("test", cfg, shuffle=False)
print(f"Test batches: {len(test_loader)}")

## Collect predictions and GT on test set

In [ ]:
all_detections = []
all_ground_truths = []
all_fnames = []  # centre-frame filenames for sequence grouping
centre = cfg.T // 2

with torch.no_grad():
    for images, targets_list, fnames_batch in test_loader:
        images = images.to(device)
        B = images.shape[0]

        centre_targets = []
        for sample_targets in targets_list:
            t = sample_targets[centre]
            centre_targets.append({
                "boxes": t["boxes"].to(device),
                "labels": t["labels"].to(device),
                "orig_size": torch.tensor([cfg.img_size, cfg.img_size], device=device),
            })

        with autocast(enabled=cfg.amp):
            outputs = model(images)

        orig_sizes = torch.stack([t["orig_size"] for t in centre_targets])
        results = postprocess(outputs, orig_sizes)

        for b in range(B):
            scores = results[b]["scores"].cpu().numpy()
            boxes = results[b]["boxes"].cpu().numpy()
            all_detections.append({"boxes": boxes, "scores": scores})
            all_fnames.append(fnames_batch[b][centre])

            gt_cxcywh = centre_targets[b]["boxes"].cpu().numpy()
            if gt_cxcywh.shape[0] > 0:
                cx, cy, w, h = gt_cxcywh[:, 0], gt_cxcywh[:, 1], gt_cxcywh[:, 2], gt_cxcywh[:, 3]
                gt_xyxy = np.column_stack([
                    (cx - w / 2) * cfg.img_size,
                    (cy - h / 2) * cfg.img_size,
                    (cx + w / 2) * cfg.img_size,
                    (cy + h / 2) * cfg.img_size,
                ])
            else:
                gt_xyxy = np.zeros((0, 4), dtype=np.float32)
            all_ground_truths.append(gt_xyxy)

print(f"Test windows: {len(all_detections)}")
total_gt = sum(g.shape[0] for g in all_ground_truths)
print(f"Total GT boxes: {total_gt}")

## Metrics

In [ ]:
def compute_recall_precision(all_detections, all_ground_truths, conf_thresh, iou_thresh):
    """Compute TP, FP, FN, precision, recall at given thresholds."""
    tp = 0
    fp = 0
    fn = 0

    for det, gt in zip(all_detections, all_ground_truths):
        mask = det["scores"] >= conf_thresh
        pred_boxes = det["boxes"][mask]
        n_gt = gt.shape[0]

        if n_gt == 0:
            fp += pred_boxes.shape[0]
            continue
        if pred_boxes.shape[0] == 0:
            fn += n_gt
            continue

        matched_gt = np.zeros(n_gt, dtype=bool)
        # Sort by descending confidence
        scores_filt = det["scores"][mask]
        order = np.argsort(-scores_filt)

        for d_box in pred_boxes[order]:
            ixmin = np.maximum(gt[:, 0], d_box[0])
            iymin = np.maximum(gt[:, 1], d_box[1])
            ixmax = np.minimum(gt[:, 2], d_box[2])
            iymax = np.minimum(gt[:, 3], d_box[3])
            iw = np.maximum(ixmax - ixmin, 0.0)
            ih = np.maximum(iymax - iymin, 0.0)
            inter = iw * ih
            det_area = (d_box[2] - d_box[0]) * (d_box[3] - d_box[1])
            gt_area = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
            union = det_area + gt_area - inter
            iou = inter / np.maximum(union, 1e-6)
            best = np.argmax(iou)
            if iou[best] >= iou_thresh and not matched_gt[best]:
                matched_gt[best] = True
                tp += 1
            else:
                fp += 1

        fn += int(np.sum(~matched_gt))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return tp, fp, fn, precision, recall

In [ ]:
from rfdetr_temporal.evaluate import evaluate_map

IOU_THRESH = 0.25

# Recall at conf > 0.15
tp_r, fp_r, fn_r, prec_r, recall_015 = compute_recall_precision(
    all_detections, all_ground_truths, conf_thresh=0.15, iou_thresh=IOU_THRESH
)
print("=" * 50)
print(f"Recall  @ conf>0.15, IoU≥{IOU_THRESH}")
print(f"  TP={tp_r}  FP={fp_r}  FN={fn_r}")
print(f"  Recall    = {recall_015:.4f}")
print(f"  Precision = {prec_r:.4f}")
print("=" * 50)

# Precision at conf > 0.25
tp_p, fp_p, fn_p, prec_025, recall_p = compute_recall_precision(
    all_detections, all_ground_truths, conf_thresh=0.25, iou_thresh=IOU_THRESH
)
print(f"Precision @ conf>0.25, IoU≥{IOU_THRESH}")
print(f"  TP={tp_p}  FP={fp_p}  FN={fn_p}")
print(f"  Precision = {prec_025:.4f}")
print(f"  Recall    = {recall_p:.4f}")
print("=" * 50)

# mAP50
map50 = evaluate_map(all_detections, all_ground_truths, iou_threshold=0.5)
print(f"mAP@0.50 = {map50:.4f}")
print("=" * 50)

In [ ]:
## Доля последовательностей с recall ≥30% кадров

In [ ]:
import re
from collections import defaultdict

# Filename parsers (same as dataset.py)
_FNAME_RE = re.compile(r"^(\d+_\d+)_(\d+)_(\d+)_bmp_jpg\.rf\.[0-9a-f]+\.jpg$")
_CADICA_RE = re.compile(r"^(p\d+)_v(\d+)_(\d+)\.(?:png|jpg)$")

def parse_fname(fname):
    m = _FNAME_RE.match(fname)
    if m:
        return (m.group(1), int(m.group(2)))
    m = _CADICA_RE.match(fname)
    if m:
        return (m.group(1), int(m.group(2)))
    return None

def frame_has_tp(det, gt, conf_thresh=0.15, iou_thresh=0.25):
    """Check if at least one GT box is matched (TP) in this frame."""
    if gt.shape[0] == 0:
        return None  # no GT — skip frame
    mask = det["scores"] >= conf_thresh
    pred_boxes = det["boxes"][mask]
    if pred_boxes.shape[0] == 0:
        return False
    matched = np.zeros(gt.shape[0], dtype=bool)
    scores_filt = det["scores"][mask]
    order = np.argsort(-scores_filt)
    for d_box in pred_boxes[order]:
        ixmin = np.maximum(gt[:, 0], d_box[0])
        iymin = np.maximum(gt[:, 1], d_box[1])
        ixmax = np.minimum(gt[:, 2], d_box[2])
        iymax = np.minimum(gt[:, 3], d_box[3])
        iw = np.maximum(ixmax - ixmin, 0.0)
        ih = np.maximum(iymax - iymin, 0.0)
        inter = iw * ih
        det_a = (d_box[2] - d_box[0]) * (d_box[3] - d_box[1])
        gt_a = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
        union = det_a + gt_a - inter
        iou = inter / np.maximum(union, 1e-6)
        best = np.argmax(iou)
        if iou[best] >= iou_thresh and not matched[best]:
            matched[best] = True
    return np.any(matched)

# Group frames by sequence
seq_frames = defaultdict(list)  # seq_key -> list of (has_tp_or_None)
for i, fname in enumerate(all_fnames):
    key = parse_fname(fname)
    if key is None:
        key = ("unknown", 0)
    has_tp = frame_has_tp(all_detections[i], all_ground_truths[i])
    seq_frames[key].append(has_tp)

# For each sequence: compute fraction of GT-frames where at least 1 TP
seq_recall_frac = {}
for key, frame_results in seq_frames.items():
    gt_frames = [r for r in frame_results if r is not None]
    if len(gt_frames) == 0:
        continue  # sequence has no GT at all
    detected = sum(1 for r in gt_frames if r)
    seq_recall_frac[key] = detected / len(gt_frames)

n_seqs = len(seq_recall_frac)
n_found = sum(1 for frac in seq_recall_frac.values() if frac >= 0.25)

print(f"Sequences with GT: {n_seqs}")
print(f"Sequences with ≥30% frames detected: {n_found}/{n_seqs} = {n_found/max(n_seqs,1):.4f}")
print()
for key in sorted(seq_recall_frac.keys()):
    frac = seq_recall_frac[key]
    status = "✓" if frac >= 0.25 else "✗"
    print(f"  {status} {key[0]}_seq{key[1]}: {frac:.1%}")

In [ ]:
print("=" * 50)
print("*** Summary (test set) ***")
print("=" * 50)
print(f"Recall      @ conf>0.15, IoU≥0.25 = {recall_015:.4f}")
print(f"Precision   @ conf>0.25, IoU≥0.25 = {prec_025:.4f}")
print(f"mAP@0.50                           = {map50:.4f}")
print(f"Seq ≥30% detected                  = {n_found}/{n_seqs} ({n_found/max(n_seqs,1):.1%})")
print(f"No-flip rate (presence)            = {cons['no_flip_rate']:.4f}")
print(f"Mean box IoU (consecutive)         = {cons['mean_box_iou']:.4f}")
print("=" * 50)

## Temporal Consistency (dataset2_split test)

Для каждой пары соседних кадров (N, N+1) где **оба имеют GT**:
- Кадр считается "detected" если хотя бы один GT бокс правильно найден (TP при conf≥0.15, IoU≥0.25)
- **Flip**: на кадре N найден (TP), а на N+1 не найден (FN), или наоборот
- **Consistency** = 1 − flips / gt_pairs

In [ ]:
def compute_temporal_consistency(all_detections, all_ground_truths, all_fnames,
                                  conf_thresh=0.15, iou_match=0.25):
    """Prediction-based temporal consistency.

    For each pair of consecutive frames (N, N+1) in a sequence:
    - Filter predictions by conf_thresh
    - "presence flip": frame N has detections, N+1 doesn't (or vice versa)
    - "box consistency": for frames where both have detections, compute max IoU
      between each box in frame N and all boxes in frame N+1.
      box_consistency = mean of max-IoUs across all boxes in both frames.

    Returns per-sequence stats + global summary.
    """
    seq_indices = defaultdict(list)
    for i, fname in enumerate(all_fnames):
        key = parse_fname(fname)
        if key is None:
            key = ("unknown", 0)
        seq_indices[key].append(i)

    total_pairs = 0
    total_flips = 0
    box_ious = []  # per-pair mean max-IoU (only when both frames have dets)
    seq_stats = {}

    for key, indices in sorted(seq_indices.items()):
        # Per-frame filtered predictions
        frame_boxes = []
        for i in indices:
            mask = all_detections[i]["scores"] >= conf_thresh
            frame_boxes.append(all_detections[i]["boxes"][mask])

        s_pairs = 0
        s_flips = 0
        s_box_ious = []

        for j in range(len(indices) - 1):
            s_pairs += 1
            has_a = frame_boxes[j].shape[0] > 0
            has_b = frame_boxes[j + 1].shape[0] > 0

            if has_a != has_b:
                s_flips += 1
            elif has_a and has_b:
                # Compute pairwise IoU between boxes in frames j and j+1
                ba = frame_boxes[j]
                bb = frame_boxes[j + 1]
                # For each box in A, max IoU with any box in B
                max_ious_a = []
                for box_a in ba:
                    ix1 = np.maximum(bb[:, 0], box_a[0])
                    iy1 = np.maximum(bb[:, 1], box_a[1])
                    ix2 = np.minimum(bb[:, 2], box_a[2])
                    iy2 = np.minimum(bb[:, 3], box_a[3])
                    iw = np.maximum(ix2 - ix1, 0.0)
                    ih = np.maximum(iy2 - iy1, 0.0)
                    inter = iw * ih
                    a_area = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
                    b_area = (bb[:, 2] - bb[:, 0]) * (bb[:, 3] - bb[:, 1])
                    union = a_area + b_area - inter
                    iou = inter / np.maximum(union, 1e-6)
                    max_ious_a.append(iou.max())
                pair_iou = float(np.mean(max_ious_a))
                s_box_ious.append(pair_iou)

        total_pairs += s_pairs
        total_flips += s_flips
        box_ious.extend(s_box_ious)

        seq_stats[key] = {
            "pairs": s_pairs,
            "flips": s_flips,
            "no_flip_rate": 1 - s_flips / max(s_pairs, 1),
            "mean_box_iou": float(np.mean(s_box_ious)) if s_box_ious else None,
        }

    return {
        "total_pairs": total_pairs,
        "total_flips": total_flips,
        "no_flip_rate": 1 - total_flips / max(total_pairs, 1),
        "mean_box_iou": float(np.mean(box_ious)) if box_ious else None,
        "per_seq": seq_stats,
    }


# --- Test set consistency (predictions only) ---
cons = compute_temporal_consistency(all_detections, all_ground_truths, all_fnames, conf_thresh=0.15)

print("=" * 60)
print("Temporal Consistency (dataset2_split test, conf≥0.15)")
print("=" * 60)
print(f"  Pairs:          {cons['total_pairs']}")
print(f"  Presence flips: {cons['total_flips']}  →  no-flip rate = {cons['no_flip_rate']:.4f}")
print(f"  Mean box IoU (consecutive frames, both have dets): {cons['mean_box_iou']:.4f}")
print()
print("Per-sequence:")
for key, s in sorted(cons["per_seq"].items()):
    tag = f"{key[0]}_seq{key[1]}"
    iou_str = f"{s['mean_box_iou']:.2f}" if s['mean_box_iou'] is not None else "n/a"
    print(f"  {tag:25s}  flips={s['flips']:3d}/{s['pairs']:3d}  no-flip={s['no_flip_rate']:.2f}  box_iou={iou_str}")

---
# Evaluation on CADICA 50+ (cadica_50plus_new)

Same model (`temporal_v1`), but evaluated on the **cadica_50plus_new** dataset — all CADICA sequences with stenosis ≥50%, single class (label 0).

In [ ]:
# Load cadica_50plus_new as "test" split
from rfdetr_temporal.dataset import TemporalStenosisDataset, collate_fn
from torch.utils.data import DataLoader
from copy import deepcopy

cfg_cadica = deepcopy(cfg)
cfg_cadica.data_root = Path("data/cadica_50plus_new")

cadica_loader = DataLoader(
    TemporalStenosisDataset("test", cfg_cadica),
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)
print(f"CADICA 50+ test batches: {len(cadica_loader)}")

In [ ]:
cadica_detections = []
cadica_ground_truths = []
cadica_fnames = []

with torch.no_grad():
    for images, targets_list, fnames_batch in cadica_loader:
        images = images.to(device)
        B = images.shape[0]

        centre_targets = []
        for sample_targets in targets_list:
            t = sample_targets[centre]
            centre_targets.append({
                "boxes": t["boxes"].to(device),
                "labels": t["labels"].to(device),
                "orig_size": torch.tensor([cfg.img_size, cfg.img_size], device=device),
            })

        with autocast(enabled=cfg.amp):
            outputs = model(images)

        orig_sizes = torch.stack([t["orig_size"] for t in centre_targets])
        results = postprocess(outputs, orig_sizes)

        for b in range(B):
            scores = results[b]["scores"].cpu().numpy()
            boxes = results[b]["boxes"].cpu().numpy()
            cadica_detections.append({"boxes": boxes, "scores": scores})
            cadica_fnames.append(fnames_batch[b][centre])

            gt_cxcywh = centre_targets[b]["boxes"].cpu().numpy()
            if gt_cxcywh.shape[0] > 0:
                cx, cy, w, h = gt_cxcywh[:, 0], gt_cxcywh[:, 1], gt_cxcywh[:, 2], gt_cxcywh[:, 3]
                gt_xyxy = np.column_stack([
                    (cx - w / 2) * cfg.img_size,
                    (cy - h / 2) * cfg.img_size,
                    (cx + w / 2) * cfg.img_size,
                    (cy + h / 2) * cfg.img_size,
                ])
            else:
                gt_xyxy = np.zeros((0, 4), dtype=np.float32)
            cadica_ground_truths.append(gt_xyxy)

print(f"CADICA 50+ windows: {len(cadica_detections)}")
total_gt_cadica = sum(g.shape[0] for g in cadica_ground_truths)
print(f"Total GT boxes: {total_gt_cadica}")

In [ ]:
# Metrics on CADICA 50+
tp_r_c, fp_r_c, fn_r_c, prec_r_c, recall_015_c = compute_recall_precision(
    cadica_detections, cadica_ground_truths, conf_thresh=0.15, iou_thresh=IOU_THRESH
)
print("=" * 50)
print(f"CADICA 50+ — Recall  @ conf>0.15, IoU≥{IOU_THRESH}")
print(f"  TP={tp_r_c}  FP={fp_r_c}  FN={fn_r_c}")
print(f"  Recall    = {recall_015_c:.4f}")
print(f"  Precision = {prec_r_c:.4f}")
print("=" * 50)

tp_p_c, fp_p_c, fn_p_c, prec_025_c, recall_p_c = compute_recall_precision(
    cadica_detections, cadica_ground_truths, conf_thresh=0.25, iou_thresh=IOU_THRESH
)
print(f"CADICA 50+ — Precision @ conf>0.25, IoU≥{IOU_THRESH}")
print(f"  TP={tp_p_c}  FP={fp_p_c}  FN={fn_p_c}")
print(f"  Precision = {prec_025_c:.4f}")
print(f"  Recall    = {recall_p_c:.4f}")
print("=" * 50)

map50_c = evaluate_map(cadica_detections, cadica_ground_truths, iou_threshold=0.5)
print(f"CADICA 50+ — mAP@0.50 = {map50_c:.4f}")
print("=" * 50)

In [ ]:
# Sequence-level recall on CADICA 50+
seq_frames_c = defaultdict(list)
for i, fname in enumerate(cadica_fnames):
    key = parse_fname(fname)
    if key is None:
        key = ("unknown", 0)
    has_tp = frame_has_tp(cadica_detections[i], cadica_ground_truths[i])
    seq_frames_c[key].append(has_tp)

seq_recall_frac_c = {}
for key, frame_results in seq_frames_c.items():
    gt_frames = [r for r in frame_results if r is not None]
    if len(gt_frames) == 0:
        continue
    detected = sum(1 for r in gt_frames if r)
    seq_recall_frac_c[key] = detected / len(gt_frames)

n_seqs_c = len(seq_recall_frac_c)
n_found_c = sum(1 for frac in seq_recall_frac_c.values() if frac >= 0.3)

print(f"CADICA 50+ sequences with GT: {n_seqs_c}")
print(f"Sequences with ≥30% frames detected: {n_found_c}/{n_seqs_c} = {n_found_c/max(n_seqs_c,1):.4f}")
print()
for key in sorted(seq_recall_frac_c.keys()):
    frac = seq_recall_frac_c[key]
    status = "✓" if frac >= 0.3 else "✗"
    print(f"  {status} {key[0]}_v{key[1]}: {frac:.1%}")

## Temporal Consistency (CADICA 50+)

In [ ]:
cons_c = compute_temporal_consistency(cadica_detections, cadica_ground_truths, cadica_fnames, conf_thresh=0.15)

print("=" * 60)
print("Temporal Consistency (CADICA 50+, conf≥0.15)")
print("=" * 60)
print(f"  Pairs:          {cons_c['total_pairs']}")
print(f"  Presence flips: {cons_c['total_flips']}  →  no-flip rate = {cons_c['no_flip_rate']:.4f}")
iou_str = f"{cons_c['mean_box_iou']:.4f}" if cons_c['mean_box_iou'] is not None else "n/a"
print(f"  Mean box IoU (consecutive frames, both have dets): {iou_str}")
print()
print("Per-sequence:")
for key, s in sorted(cons_c["per_seq"].items()):
    tag = f"{key[0]}_v{key[1]}"
    iou_s = f"{s['mean_box_iou']:.2f}" if s['mean_box_iou'] is not None else "n/a"
    print(f"  {tag:25s}  flips={s['flips']:3d}/{s['pairs']:3d}  no-flip={s['no_flip_rate']:.2f}  box_iou={iou_s}")

In [ ]:
print("=" * 50)
print("*** CADICA 50+ Summary ***")
print("=" * 50)
print(f"Recall      @ conf>0.15, IoU≥0.25 = {recall_015_c:.4f}")
print(f"Precision   @ conf>0.25, IoU≥0.25 = {prec_025_c:.4f}")
print(f"mAP@0.50                           = {map50_c:.4f}")
print(f"Seq ≥30% detected                  = {n_found_c}/{n_seqs_c} ({n_found_c/max(n_seqs_c,1):.1%})")
iou_str_c = f"{cons_c['mean_box_iou']:.4f}" if cons_c['mean_box_iou'] is not None else "n/a"
print(f"No-flip rate (presence)            = {cons_c['no_flip_rate']:.4f}")
print(f"Mean box IoU (consecutive)         = {iou_str_c}")
print("=" * 50)